In [ ]:
import numpy as np
import mne
import matplotlib.pyplot as plt
import glob
import os
import seaborn as sns
import matplotlib.dates as mdates
import pandas as pd
import matplotlib.gridspec as gridspec
import matplotlib.transforms

epoch_length = 60*5

#file specifications (Nihon Kohden and Natus are split)
folders = ['01-18-2023','03-07-2024','03-27-2024','05-22-2024','07-23-2024','08-01-2024','08-02-2024','08-03-2024','11-14-2024','02-05-2025']
ignore = ['DA1360YE', 'DA1360YF', 'DA1360YG', 'DA1360YN', 'DA1360YO', 'DA1360YP', 'DA1360YQ','GA42712E', 'DA4440KZ', 'DA4440L3']

for folder in folders:
    filelist = glob.glob(f'../data/eeg/{folder}/EEG2100/*.eeg') #Nihon Kohden
    filelist = glob.glob(f'../data/{folder}/EEG2100/*.edf') #Natus
    raw_list = []
    for path in filelist:
        filename = os.path.basename(path)
        fn = filename[:-4]
        #if file doesn't exist, create it, otherwise skip
        if fn in ignore:
            print(f"Skipping {fn} because it is in the ignore list")
        else:
            #raw = mne.io.read_raw_nihon(path,preload=True)
            raw = mne.io.read_raw_edf(path,preload=True)
            n_time_samps = raw.n_times
            time_secs = raw.times
            ch_names = raw.ch_names
            sfreq = raw.info['sfreq']
            n_chan = len(ch_names)  # note: there is no raw.n_channels attribute
            print(f"Total duration is {n_time_samps / sfreq} sec")
            #Set channel types
            ch_names = raw.ch_names
            exclude_ch = ['EEG A1-Ref','EEG A2-Ref','EEG Fpz-Ref','EEG L EOG-Ref', 'EEG R EOG-Ref', 'EEG ECG-Ref', 'EEG RTEMP-Ref', 'EEG LFTEMP-Ref', 'EEG LIMB-Ref', 'EEG BODY-Ref', 'EEG RESP1-Ref', 'EEG RESP2-Ref', 'EEG T1-Ref', 'EEG T2-Ref', 'EEG RL1-Ref', 'EEG RL2-Ref', 'EEG X14-Ref', 'EEG X15-Ref', 'EEG X16-Ref', 'EEG X17-Ref', 'EEG X18-Ref', 'DC1', 'DC2', 'DC3', 'DC4', 'OSAT', 'PR', 'EEG Event']
            raw.info['bads'] = exclude_ch
            reject_criteria = dict(
                eeg=100e-6,  # 100 µV
            ) 
            flat_criteria = dict( eeg=1e-6)   

            #set channel types
            channel_types = {ch: 'misc' for ch in exclude_ch}
            raw.set_channel_types(channel_types)

            #remove prefix of "EEG" and suffix of "-Ref" from channel names
            new_ch_names = {ch: ch[4:-4] if ch.startswith('EEG') and ch.endswith('-Ref') else ch for ch in ch_names}
            raw.rename_channels(new_ch_names)     

            #create copy, load seizure data, filter, then chunk
            #leave out first and last minute of data
            rawcc=  raw.copy().crop(tmin= 60, tmax = (n_time_samps / sfreq) - 60)
            
            #BAD SEGMENT ANNOTATION
            rawbp= rawcc.copy()
            anode_channels = ['Fp1', 'F7', 'T3', 'T5', 'Fp1', 'F3', 'C3', 'P3', 'Fz', 'Cz','Fp2', 'F4', 'C4', 'P4', 'Fp2', 'F8', 'T4', 'T6']
            cathode_channels = ['F7', 'T3', 'T5', 'O1', 'F3', 'C3', 'P3', 'O1', 'Cz', 'Pz', 'F4', 'C4', 'P4', 'O2', 'F8', 'T4', 'T6','O2']
            new_ch_names = ['Fp1-F7', 'F7-T3', 'T3-T5', 'T5-O1', 'Fp1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'Fz-Cz', 'Cz-Pz', 'Fp2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'Fp2-F8', 'F8-T4', 'T4-T6', 'T6-O2']
            # Set bipolar reference
            rawbp = mne.set_bipolar_reference(rawbp, anode=anode_channels, cathode=cathode_channels, ch_name=new_ch_names, verbose=True)

            # Filter data with bandpass filter from 1 to 70 hz
            rawbp.filter(1, 70, fir_design='firwin')

            # Notch Filter at 60 hz
            rawbp.notch_filter(60, fir_design='firwin')

            #Remove any bad times/disconnections
            bad_annot_fn= f'{folder}-{fn}-badannots.csv'
            slp_annot_fn= f'{folder}-{fn}-sleepannots.csv'

            #Manual labeling of sleep and bad segments of data to exclude
            if os.path.exists(slp_annot_fn):
                print(f"Loading sleep and bad segment annotations from {slp_annot_fn}")
                slp_annot = mne.read_annotations(slp_annot_fn)
                print(slp_annot)
                #check if there are any annotations in slp_annot with description 'sleep'
                if 'sleep' in slp_annot.description:
                    slp_annot.rename({'sleep':'bad_sleep'})
                rawcc.set_annotations(slp_annot)
                rawbp.set_annotations(slp_annot)
                #rawbp.plot(block=True)
                #val = input("Enter Y to save new annotations: ")
                #if val=='Y':    
                #    rawbp.annotations.save(slp_annot_fn, overwrite=True)
                #    rawcc.set_annotations(rawbp.annotations)
            else:
                #print(f"Loading bad segment annotations from {bad_annot_fn}")
                #bad_annot = mne.read_annotations(bad_annot_fn)
                #rawbp.set_annotations(bad_annot)
                print('Stop here to plot and save any annotations')
                rawbp.plot(block=True)
                val = input("Enter Y to save new annotations: ")
                if val=='Y':    
                    rawbp.annotations.save(slp_annot_fn, overwrite=True)
                    rawcc.set_annotations(rawbp.annotations)
            raw_list.append(rawcc)

    print(folder)
    raw = mne.concatenate_raws(raw_list)

    montage = mne.channels.make_standard_montage("standard_1020")
    #print(montage)
    raw.set_montage(montage,on_missing='warn')

    #rawcc.load_data()
    # Filter data with bandpass filter from 1 to 70 hz
    raw.filter(1, 70, fir_design='firwin')

    # Notch Filter at 60 hz
    raw.notch_filter(60, fir_design='firwin')

    #COMPUTE PSD 
    epochs = mne.make_fixed_length_epochs(raw, duration=epoch_length, preload=False)
    epo_spectrum = epochs.compute_psd(fmax=sfreq/2,exclude='bads')
    psds, freqs = epo_spectrum.get_data(return_freqs=True)
    print(f"\npsds_agg_agg shape: {psds.shape}, freqs shape: {freqs.shape}")

    #SAVE TEMPORARY FILES
    #np.savez_compressed(f'{folder}-{epoch_length}-slpomitted.npz', name1=psds, name2=freqs) 

In [ ]:
delta_power = []
theta_power = []
alpha_power = []
beta_power = []
gamma_power = []
ad_ratio_power = []
rav_power = []

epoch_length = 60*5
#folders = ['01-18-2023','03-07-2024','03-27-2024','05-22-2024','07-23-2024','08-01-2024','11-14-2024','02-05-2025']

for folder in folders:
    dat = np.load(f'{folder}-{epoch_length}-slpomitted.npz')
    psds_agg = dat['name1']
    freqs = dat['name2']

    delta = np.sum(psds_agg[:, :, (freqs >= 1) & (freqs <= 4)], axis=-1)/np.sum(psds_agg, axis=-1)
    theta = np.sum(psds_agg[:, :, (freqs >= 4) & (freqs <= 8)], axis=-1)/np.sum(psds_agg, axis=-1)
    alpha = np.sum(psds_agg[:, :, (freqs >= 8) & (freqs <= 13)], axis=-1)/np.sum(psds_agg, axis=-1)
    beta = np.sum(psds_agg[:, :, (freqs >= 13) & (freqs <= 30)], axis=-1)/np.sum(psds_agg, axis=-1)
    gamma = np.sum(psds_agg[:, :, (freqs >= 30) & (freqs <= 50)], axis=-1)/np.sum(psds_agg, axis=-1)
    ad_ratio = delta/alpha
    rav = np.sum(psds_agg[:, :, (freqs >= 8) & (freqs <= 13)], axis=-1)/np.sum(psds_agg[:, :, (freqs >= 1) & (freqs <= 20)], axis=-1)

    #average over channels for each band 
    delta_avg_over_ch = np.mean(delta, axis=1)
    theta_avg_over_ch = np.mean(theta, axis=1)
    alpha_avg_over_ch = np.mean(alpha, axis=1)
    beta_avg_over_ch = np.mean(beta, axis=1)
    gamma_avg_over_ch = np.mean(gamma, axis=1)
    ad_ratio_avg_over_ch = np.mean(ad_ratio, axis=1)
    rav_avg_over_ch = np.mean(rav, axis=1)
    #print(f"\nDelta shape: {delta_avg.shape}")

    #store average power for each band
    delta_power.append(delta_avg_over_ch)
    theta_power.append(theta_avg_over_ch)
    alpha_power.append(alpha_avg_over_ch)
    beta_power.append(beta_avg_over_ch)
    gamma_power.append(gamma_avg_over_ch)
    ad_ratio_power.append(ad_ratio_avg_over_ch)
    rav_power.append(rav_avg_over_ch)

#create pandas dataframe for each power band with file name as index
df_dict = {}
df_dict['Delta'] = pd.DataFrame(delta_power, index=folders)
df_dict['Theta'] = pd.DataFrame(theta_power, index=folders)
df_dict['Alpha'] = pd.DataFrame(alpha_power, index=folders)
df_dict['Beta'] = pd.DataFrame(beta_power, index=folders)
df_dict['Gamma'] = pd.DataFrame(gamma_power, index=folders)
df_dict['AD Ratio'] = pd.DataFrame(ad_ratio_power, index=folders)
df_dict['RAV'] = pd.DataFrame(rav_power, index=folders)   

In [ ]:
import matplotlib as mpl
# Set global matplotlib parameters for publication quality
mpl.rcParams.update({
    'font.size': 14,
    'axes.labelsize': 16,
    'axes.titlesize': 18,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 12,
    'figure.dpi': 300,
    'savefig.dpi': 600,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.linewidth': 1.2,
    'figure.figsize': (20, 4.5),
    'boxplot.flierprops.marker': 'o',
    'boxplot.flierprops.markersize': 3,
    'boxplot.flierprops.markerfacecolor': 'gray'
})

# Violin plots
fig, ax = plt.subplots(1, len(df_dict), figsize=(20, 4.5))
for i, key in enumerate(df_dict):
    test = df_dict[key].stack().reset_index()
    test.columns=['Date','','Power']
    sns.violinplot(ax=ax[i], x='Date', y='Power', data=test, inner='box', linewidth=1.2)
    ax[i].set_title(f'{key}', fontsize=16)
    ax[i].set_xlabel('')
    ax[i].set_ylabel('Relative Power' if i == 0 else '')
    ax[i].tick_params(axis='x', rotation=30)
    ax[i].grid(True, linestyle='--', alpha=0.5)
fig.suptitle('EEG Metrics over time', fontsize=20, y=1.05)
fig.tight_layout()
fig.savefig('violinplots-slpomitted-publication.pdf', bbox_inches='tight')
plt.show()

In [ ]:
import statsmodels.formula.api as smf

# Prepare results dictionary
mixedlm_results = {}

# Loop through each metric in df_dict_day
for metric, df_metric in df_dict_day.items():
    # Stack to long format
    df_long = df_metric.stack().reset_index()
    df_long.columns = ['Session', 'Epoch', 'Power']
    # Fit mixed effects model: Session as fixed, Epoch as random (nested within Session)
    # Here, 'Session' is the session label (0-7), 'Epoch' is repeated within session
    model = smf.mixedlm("Power ~ C(Session)", df_long, groups=df_long["Session"])
    result = model.fit()
    mixedlm_results[metric] = result
    print(f"\n=== {metric} ===")
    print(result.summary())

# Save mixed effects model results to a table for publication
import pandas as pd
# Create a DataFrame to store the results
results_table = pd.DataFrame(columns=['Metric', 'Fixed Effects', 'Random Effects', 'P-Value'])
# Populate the DataFrame with results
for metric, result in mixedlm_results.items():
    fixed_effects = result.fe_params.to_dict()
    random_effects = result.random_effects
    # Collect all p-values for fixed effects except the Intercept
    p_values = {k: v for k, v in result.pvalues.items() if k != 'Intercept'}
    results_table = results_table._append({
        'Metric': metric,
        'Fixed Effects': fixed_effects,
        'Random Effects': random_effects,
        'P-Value': p_values
    }, ignore_index=True)
# Save the results table to an Excel file for publication
results_table.to_excel('mixedlm_results_slpomitted.xlsx', index=False)


In [ ]:
#create another dictionary with only the power bands from df_dict
df_power = {key: df_dict[key] for key in df_dict if key in ['Delta', 'Theta', 'Alpha', 'Beta', 'Gamma']}
fig = plt.figure(figsize=(20, 4))
outer = gridspec.GridSpec(1,5, wspace=0.2, hspace=0.5)
for i, key in enumerate(df_power):
    # Prepare data
    data = df_power[key].stack().reset_index()
    data.columns = ['Date', '', 'Power']
    data['Date'] = pd.to_datetime(data['Date'], format='%m-%d-%Y')
    inner = gridspec.GridSpecFromSubplotSpec(1, 2, subplot_spec=outer[i], width_ratios=(1, 10), wspace=0.1)
    for j in range(2):
        ax1 = plt.Subplot(fig,inner[j])
        #fig,(ax1, ax2) = plt.subplots(ncols=2, sharey=True, gridspec_kw={'width_ratios': (1, 13)})
        #fig.subplots_adjust(wspace=0.05)

        sns.violinplot(ax=ax1,x='Date', y='Power', data=data, inner='box', linewidth=1,width=3,native_scale=True)

        fmt_half_year1 = mdates.MonthLocator(interval=2)
        ax1.xaxis.set_major_locator(fmt_half_year1)

        ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

        ax1.format_xdata = mdates.DateFormatter('%Y-%m')
        ax1.grid(True)

        d = 2.  # proportion of vertical to horizontal extent of the slanted line
        kwargs = dict(marker=[(-1, -d), (1, d)], markersize=10, linestyle="none", color='k', mec='k', mew=1, clip_on=False)
        if j == 0:
            ax1.spines.right.set_visible(False)
            ax1.xaxis.tick_bottom()
            datemin1 = np.datetime64('2023-01')
            datemax1 = np.datetime64('2023-02')
            ax1.set_xlim(datemin1, datemax1)

            # Define new tick locations and labels
            #new_tick_locations = [0]
            new_tick_labels = ['0']
            #ax1.set_xticks(new_tick_locations)
            ax1.set_xticklabels(new_tick_labels)

            # Create offset transform by 5 points in x direction
            dx = -10/72.; dy = 0/72. 
            offset = matplotlib.transforms.ScaledTranslation(dx, dy, fig.dpi_scale_trans)

            # apply offset transform to all x ticklabels.
            for label in ax1.xaxis.get_majorticklabels():
                label.set_transform(label.get_transform() + offset)


            ax1.plot([1, 1], [0, 1], transform=ax1.transAxes, **kwargs)
            #Set y-label only for the first and fifth plot
            if i == 0:
                ax1.set_ylabel('Relative Power')
            else:
                ax1.set_ylabel('')
                
        else:
            ax1.spines.left.set_visible(False)
            ax1.yaxis.tick_right()
            ax1.tick_params(labelright=False)
            datemin2 = np.datetime64('2024-02')
            datemax2 = np.datetime64('2025-03')
            ax1.set_xlim(datemin2, datemax2)

            # Define new tick locations and labels
            new_tick_labels = ['14', '16', '18', '20', '22', '24','26']
            ax1.set_xticklabels(new_tick_labels)

            ax1.plot([0, 0], [0, 1], transform=ax1.transAxes, **kwargs)
            ax1.set_ylabel('')
            ax1.set_title(f'{key}', fontsize=14,x=0.4)
        ax1.set_xlabel('')
        fig.add_subplot(ax1)
        #fig.autofmt_xdate()
fig.text(s='Month', x=0.5, y=-0.05)
fig.suptitle('Power Metrics over time', fontsize=20, y=1.05)
fig.tight_layout()
fig.savefig('A-PowerMetrics.pdf', bbox_inches='tight')
plt.show()

In [ ]:

#create another dictionary with only AD Ratio, RAV
df_power = {key: df_dict[key] for key in df_dict if key in ['AD Ratio', 'RAV']}
fig = plt.figure(figsize=(10,4))
outer = gridspec.GridSpec(1,2, wspace=0.2, hspace=0.5)
for i, key in enumerate(df_power):
    # Prepare data
    data = df_power[key].stack().reset_index()
    data.columns = ['Date', '', 'Power']
    data['Date'] = pd.to_datetime(data['Date'], format='%m-%d-%Y')
    inner = gridspec.GridSpecFromSubplotSpec(1, 2, subplot_spec=outer[i], width_ratios=(1, 10), wspace=0.1)
    for j in range(2):
        ax1 = plt.Subplot(fig,inner[j])
        #fig,(ax1, ax2) = plt.subplots(ncols=2, sharey=True, gridspec_kw={'width_ratios': (1, 13)})
        #fig.subplots_adjust(wspace=0.05)

        sns.violinplot(ax=ax1,x='Date', y='Power', data=data, inner='box', linewidth=1,width=3,native_scale=True)

        fmt_half_year1 = mdates.MonthLocator(interval=2)
        ax1.xaxis.set_major_locator(fmt_half_year1)

        ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

        ax1.format_xdata = mdates.DateFormatter('%Y-%m')
        ax1.grid(True)

        d = 2.  # proportion of vertical to horizontal extent of the slanted line
        kwargs = dict(marker=[(-1, -d), (1, d)], markersize=10, linestyle="none", color='k', mec='k', mew=1, clip_on=False)
        if j == 0:
            ax1.spines.right.set_visible(False)
            ax1.xaxis.tick_bottom()
            datemin1 = np.datetime64('2023-01')
            datemax1 = np.datetime64('2023-02')
            ax1.set_xlim(datemin1, datemax1)

            # Define new tick locations and labels
            #new_tick_locations = [0]
            new_tick_labels = ['0']
            #ax1.set_xticks(new_tick_locations)
            ax1.set_xticklabels(new_tick_labels)

            # Create offset transform by 5 points in x direction
            dx = -10/72.; dy = 0/72. 
            offset = matplotlib.transforms.ScaledTranslation(dx, dy, fig.dpi_scale_trans)

            # apply offset transform to all x ticklabels.
            for label in ax1.xaxis.get_majorticklabels():
                label.set_transform(label.get_transform() + offset)

            #for tick in ax1.xaxis.get_major_ticks():
            #    tick.label.set_position((-1, 0))

            ax1.plot([1, 1], [0, 1], transform=ax1.transAxes, **kwargs)
            #Set y-label only for the first and fifth plot
            if i == 0:
                ax1.set_ylabel('Ratio')
            else:
                ax1.set_ylabel('')
                
        else:
            ax1.spines.left.set_visible(False)
            ax1.yaxis.tick_right()
            ax1.tick_params(labelright=False)
            datemin2 = np.datetime64('2024-02')
            datemax2 = np.datetime64('2025-03')
            ax1.set_xlim(datemin2, datemax2)

            # Add asterisk for 2025-02-05 for RAV
            if key == 'RAV':
                y_max = data[data['Date'] == pd.to_datetime('2025-02-05')]['Power'].max()
                x_pos = list(data['Date'].unique()).index(pd.to_datetime('2025-02-05'))
                ax1.annotate('*', xy=(pd.to_datetime('2025-02-05'), y_max + 0.03), xytext=(pd.to_datetime('2025-02-05'), y_max + 0.07),
            ha='center', va='bottom', fontsize=28, color='black', fontweight='bold')
                
            # Define new tick locations and labels
            new_tick_labels = ['14', '16', '18', '20', '22', '24','26']
            ax1.set_xticklabels(new_tick_labels)

            ax1.plot([0, 0], [0, 1], transform=ax1.transAxes, **kwargs)
            ax1.set_ylabel('')
            ax1.set_title(f'{key}', fontsize=14,x=0.4)

        

        ax1.set_xlabel('')
        fig.add_subplot(ax1)
        #fig.autofmt_xdate()
fig.text(s='Month', x=0.5, y=-0.05)
fig.suptitle('AD Ratio and RAV', fontsize=20, y=1.05)
fig.tight_layout()
fig.savefig('B-ADRRAVE.pdf', bbox_inches='tight')
plt.show()